# NovaMart Data Preparation

**Student Name:** Muhammed Anshif K

**Version:** 2 – Regional Performance & Monthly Trend

In [22]:
import pandas as pd
import numpy as np

## Step 1: Read Monthly CSV Files

Load all six monthly order files into separate DataFrames.

In [23]:
jan = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_01.csv")

feb = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_02.csv")

mar = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_03.csv")

apr = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_04.csv")

may = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_05.csv")

jun = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\monthly_orders\novamart_orders_2024_06.csv")

## Step 2: Combine Monthly Data

Combine all six monthly DataFrames into one DataFrame.

In [24]:
df = pd.concat(
    [jan, feb, mar, apr, may, jun],
    ignore_index=True
)
df.head()

,order_id,order_date,ship_date,customer_id,product_id,region,category,sub_category,quantity,sales,discount,profit
0,NM202401461113,2024-01-27,2024-02-01,CU0049,PR522,central,office supplies,Storage,1,$41.69,0.00,19.67
1,NM202401729507,2024-01-29,2024-01-31,CU0229,PR507,CENTRAL,Technology,Machines,4,$170.76,0.00,91.36
2,NM202401472052,2024-01-08,2024-01-13,CU0108,PR501,east,technology,Phones,1,$728.04,0.30,14.63
3,NM202401384236,2024-01-22,2024-01-27,CU0220,PR503,CENTRAL,Technology,Phones,7,$921.60,0.15,NaN
4,NM202401487969,"Jan 04, 2024","Jan 07, 2024",CU0213,PR501,SOUTH,technology,Phones,2,"$2,080.12",0.00,653.30


## Step 3: Remove Duplicate Records

Remove fully duplicated rows from the dataset.

In [25]:
df = df.drop_duplicates()

df.shape

(6059, 12)

## Step 4: Clean Sales Column

Remove "$" and commas from the sales column and convert it to float.

In [26]:
df["sales"] = (
    df["sales"]
    .str.replace("$","",regex=False)
    .str.replace(",","",regex=False)
    .astype(float)
)

df["sales"].head()

0      41.69
1     170.76
2     728.04
3     921.60
4    2080.12
Name: sales, dtype: float64

## Step 5: Convert Date Columns

Convert order_date and ship_date into datetime format.

In [27]:
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    errors="coerce"
)

## Step 6: Standardize Text Columns

Clean category and region names.

In [28]:
df["category"] = df["category"].str.strip().str.title()

df["region"] = df["region"].str.strip().str.title()

## Step 7: Handle Missing Values

Replace blank discount with 0 and fill missing profit values.

In [29]:
df["discount"] = df["discount"].fillna(0)

df["profit"] = df["profit"].fillna(0)

## Step 8: Create New Columns

Create order month, profit margin percentage, and shipping days.

In [30]:
df["order_month"] = df["order_date"].dt.month_name()

df["profit_margin_pct"] = (
    df["profit"] /
    df["sales"]
) * 100

df["shipping_days"] = (
    df["ship_date"] -
    df["order_date"]
).dt.days

## Step 9: Merge Lookup Tables

Merge product and customer lookup tables.

In [31]:
product = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\product_lookup.csv")

customer = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\customer_lookup.csv")


df = df.merge(
    product,
    on="product_id",
    how="left"
)

df = df.merge(
    customer,
    on="customer_id",
    how="left"
)

## Step 10: Save the Clean Dataset

Save the transformed dataset for the Streamlit dashboard.

In [36]:
df.to_csv(
    r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\Output\novamart_clean.csv",
    index=False
)

print("Data saved successfully.")

Data saved successfully.


## Step 11: Verify the Clean Dataset

Load the saved `novamart_clean.csv` file and verify its structure, dimensions, data types, and missing values before building the Streamlit dashboard.

In [33]:
clean_df = pd.read_csv(r"C:\Users\dbila\Project_5.1\Project_2_NovaMart\data\novamart_clean.csv")

clean_df.head()

,order_id,order_date,ship_date,customer_id,product_id,region_x,category_x,sub_category_x,quantity,sales,...,shipping_days,product_name,category_y,sub_category_y,unit_cost,list_price,customer_name,segment,region_y,city
0,NM202401461113,2024-01-27,2024-02-01,CU0049,PR522,Central,Office Supplies,Storage,1,41.69,...,5.0,File Cabinet,Office Supplies,Storage,22.02,41.69,Sam Bell,Corporate,Central,Dallas
1,NM202401729507,2024-01-29,2024-01-31,CU0229,PR507,Central,Technology,Machines,4,170.76,...,2.0,LaserJet 100,Technology,Machines,19.85,42.69,Drew Bell,Corporate,Central,Chicago
2,NM202401472052,2024-01-08,2024-01-13,CU0108,PR501,East,Technology,Phones,1,728.04,...,5.0,Galaxy S,Technology,Phones,713.41,1040.06,Sam Dean,Consumer,East,New York
3,NM202401384236,2024-01-22,2024-01-27,CU0220,PR503,Central,Technology,Phones,7,921.60,...,5.0,iView 11,Technology,Phones,98.09,154.89,Drew Dean,Consumer,Central,Dallas
4,NM202401487969,NaN,NaN,CU0213,PR501,South,Technology,Phones,2,2080.12,...,NaN,Galaxy S,Technology,Phones,713.41,1040.06,Drew Wood,Consumer,South,Atlanta


## Step 12: Check Dataset Shape and Information

Display the number of rows and columns and inspect the data types of the cleaned dataset.

In [34]:
print("Dataset shape:", clean_df.shape)

clean_df.info()

Dataset shape: (6059, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6059 entries, 0 to 6058
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           6059 non-null   object 
 1   order_date         1999 non-null   object 
 2   ship_date          1999 non-null   object 
 3   customer_id        6059 non-null   object 
 4   product_id         6059 non-null   object 
 5   region_x           6059 non-null   object 
 6   category_x         6059 non-null   object 
 7   sub_category_x     6059 non-null   object 
 8   quantity           6059 non-null   int64  
 9   sales              6059 non-null   float64
 10  discount           6059 non-null   float64
 11  profit             6059 non-null   float64
 12  order_month        1999 non-null   object 
 13  profit_margin_pct  6059 non-null   float64
 14  shipping_days      1999 non-null   float64
 15  product_name       6059 non-null   object 
 16

## Step 13: Check Missing Values

Check whether any columns in the cleaned dataset still contain missing values.

In [35]:
clean_df.isnull().sum()

order_id                0
order_date           4060
ship_date            4060
customer_id             0
product_id              0
region_x                0
category_x              0
sub_category_x          0
quantity                0
sales                   0
discount                0
profit                  0
order_month          4060
profit_margin_pct       0
shipping_days        4060
product_name            0
category_y              0
sub_category_y          0
unit_cost               0
list_price              0
customer_name           0
segment                 0
region_y                0
city                    0
dtype: int64